# 05. 앙상블 & 최종 제출

**앙상블(Ensemble)**: 여러 모델의 예측을 합쳐 하나의 예측을 만드는 기법.
캐글 상위권 솔루션은 거의 전부 앙상블이다.

### 왜 효과가 있나
모델마다 **틀리는 지점이 다르다.**
LightGBM이 놓친 패턴을 CatBoost가 잡고, 그 반대도 일어난다.
평균을 내면 **각자의 실수가 서로 상쇄**되어 개별 모델보다 안정적인 예측이 된다.

> 핵심 조건: **모델들이 서로 달라야 한다.**
> 똑같은 모델 3개를 평균 내면 아무 효과가 없다.
> 이것을 **다양성(diversity)** 이라고 하며, 이 노트북에서 실제로 측정해본다.

### 사용할 모델 3종
| 모델 | 특징 | 다양성 확보 포인트 |
|---|---|---|
| **LightGBM** | 잎 단위(leaf-wise) 성장, 빠름 | 04단계 튜닝 결과 사용 |
| **XGBoost** | 깊이 단위(depth-wise) 성장 | 트리 성장 방식이 다름 |
| **CatBoost** | 순서형 부스팅, 대칭 트리 | 과적합 억제 방식이 근본적으로 다름 |

## 0. 준비

In [ ]:
import sys, os, json, time
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 220)

SEED = 42
np.random.seed(SEED)

from preprocess import load_data, build_final, NUM_COLS, CAT_COLS, TARGET
from cv import get_folds, run_cv

train, test, sub = load_data('../data/')
y = train[TARGET].values

# 04단계에서 확정한 최종 16피처 — 세 모델 모두 같은 피처를 쓴다
X, X_test, use_cols = build_final(train, test)
folds = get_folds(X, y, n_splits=5, seed=SEED)

print(f'X {X.shape} / X_test {X_test.shape}')
print(f'피처 {len(use_cols)}개')
print(f'lightgbm {lgb.__version__} / xgboost {xgb.__version__}')

In [ ]:
# 04단계에서 저장한 튜닝 결과를 불러온다.
# 파일이 없으면(04를 안 돌렸으면) 기본값으로 진행한다.
PARAM_PATH = '../preds/lgbm_best_params.json'
if os.path.exists(PARAM_PATH):
    with open(PARAM_PATH, encoding='utf-8') as f:
        best_params = json.load(f)
    print('04단계 튜닝 파라미터 로드:')
    for k, v in best_params.items():
        print(f'  {k:20s} {v}')
else:
    best_params = dict(num_leaves=63, min_child_samples=50,
                       feature_fraction=0.8, bagging_fraction=0.8,
                       lambda_l1=1e-8, lambda_l2=1e-8)
    print('⚠️ 04단계 결과 파일이 없어 기본값을 사용합니다')

## 1. 모델 1 — LightGBM

04단계에서 튜닝한 파라미터를 그대로 쓴다.

In [ ]:
LGB_PARAMS = dict(objective='binary', metric='auc', verbose=-1, n_jobs=-1,
                  random_state=SEED, n_estimators=4000, learning_rate=0.05,
                  bagging_freq=1, **best_params)
LGB_FIT = dict(eval_metric='auc',
               callbacks=[lgb.early_stopping(100, verbose=False)])

# ⏱ 약 5~8분
res_lgb = run_cv(lambda: lgb.LGBMClassifier(**LGB_PARAMS),
                 X, y, X_test, folds=folds, name='LightGBM', fit_params=LGB_FIT)

## 2. 모델 2 — XGBoost

### LightGBM과 무엇이 다른가
트리를 키우는 **방향**이 다르다.

- **LightGBM (leaf-wise)**: 손실이 가장 많이 줄어드는 **잎 하나**를 골라 쪼갠다.
  → 깊고 비대칭적인 트리. 빠르고 강력하지만 과적합 위험이 크다.
- **XGBoost (depth-wise)**: 같은 깊이의 **모든 잎**을 한 층씩 쪼갠다.
  → 균형 잡힌 트리. 보수적이라 다른 종류의 실수를 한다.

이 **구조적 차이**가 앙상블에 필요한 다양성을 만든다.

> `enable_categorical=True` + `tree_method='hist'` 조합으로
> XGBoost도 pandas의 category dtype을 직접 처리할 수 있다.

In [ ]:
XGB_PARAMS = dict(
    objective='binary:logistic', eval_metric='auc',
    tree_method='hist',          # 히스토그램 기반 분할 (대용량에서 빠름)
    enable_categorical=True,     # category dtype 직접 처리
    max_depth=8,                 # depth-wise 성장이라 깊이로 복잡도를 제어
    learning_rate=0.05,
    subsample=0.8,               # LightGBM의 bagging_fraction에 해당
    colsample_bytree=0.8,        # LightGBM의 feature_fraction에 해당
    min_child_weight=20,
    reg_lambda=1.0,
    n_estimators=3000,
    n_jobs=-1, random_state=SEED,
    early_stopping_rounds=100,   # XGBoost는 생성자에서 지정한다
    verbosity=0,
)

# ⏱ 약 5~10분
res_xgb = run_cv(lambda: xgb.XGBClassifier(**XGB_PARAMS),
                 X, y, X_test, folds=folds, name='XGBoost',
                 fit_params=dict(verbose=False))

## 3. 모델 3 — CatBoost

### 무엇이 다른가
**순서형 부스팅(Ordered Boosting)** 이라는 독자적인 기법을 쓴다.

일반 부스팅은 "모든 데이터로 계산한 잔차"를 다음 트리가 학습하는데,
이 과정에서 **자기 자신의 정답을 살짝 엿보는** 편향이 생긴다(target leakage).
CatBoost는 데이터를 **인위적인 시간 순서**로 배열하고,
각 시점에서 **그 이전 데이터만으로** 잔차를 계산해 이 편향을 없앤다.

또 **대칭 트리(oblivious tree)** 를 쓴다. 같은 깊이에서는 모든 노드가
**같은 분기 조건**을 쓰는 구조라, 강한 규제 효과가 있고 예측이 매우 빠르다.

> CatBoost는 결측 범주형을 문자열로 받아야 하므로, 여기서만 `'NA'`로 채운다.
> (수치형 결측은 CatBoost가 알아서 처리한다)

In [ ]:
# CatBoost용 데이터 준비 — 범주형만 문자열로 변환
#
# ⚠️ pandas 3.0 주의: category dtype에 .astype(str)을 하면 결측이 문자열 'nan'이
#    되는 게 아니라 NA 상태 그대로 남는다. CatBoost는 범주형에 NA를 허용하지 않아
#    "cat_features must be integer or string" 에러가 난다.
#    그래서 np.where로 결측을 먼저 'NA' 문자열로 확실히 바꾼다.
def to_catboost(df):
    out = df.copy()
    for c in CAT_COLS:
        s = out[c]
        out[c] = pd.Series(np.where(s.isna(), 'NA', s.astype(str)),
                           index=out.index).astype(str)
    return out

X_cb, X_test_cb = to_catboost(X), to_catboost(X_test)
cat_idx = [X_cb.columns.get_loc(c) for c in CAT_COLS]

print('범주형 열 인덱스:', cat_idx)
print(X_cb[CAT_COLS].head(3).to_string())

In [ ]:
CB_PARAMS = dict(
    loss_function='Logloss', eval_metric='AUC',
    iterations=3000, learning_rate=0.05,
    depth=8,                     # 대칭 트리의 깊이
    l2_leaf_reg=3.0,
    random_seed=SEED,
    cat_features=cat_idx,        # 어느 열이 범주형인지 알려준다
    early_stopping_rounds=100,
    verbose=0, thread_count=-1,
    allow_writing_files=False,   # 학습 로그 파일 생성 방지
)

# ⏱ 약 10~20분 (셋 중 가장 느리다)
res_cb = run_cv(lambda: CatBoostClassifier(**CB_PARAMS),
                X_cb, y, X_test_cb, folds=folds, name='CatBoost',
                fit_params=dict(verbose=0))

## 4. 개별 모델 성능 비교

In [ ]:
models = {'LightGBM': res_lgb, 'XGBoost': res_xgb, 'CatBoost': res_cb}

perf = pd.DataFrame([{
    '모델': k,
    'CV AUC': v['cv_auc'],
    '폴드 평균': v['fold_mean'],
    '폴드 표준편차': v['fold_std'],
    '학습시간(초)': round(v['elapsed'], 1),
} for k, v in models.items()]).sort_values('CV AUC', ascending=False)
print(perf.round(6).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
d = perf.sort_values('CV AUC')
ax.barh(d['모델'], d['CV AUC'], color='#457b9d',
        xerr=d['폴드 표준편차'], capsize=4)
ax.set_xlim(d['CV AUC'].min() - 0.002, d['CV AUC'].max() + 0.001)
ax.set_xlabel('CV AUC (오차막대 = 폴드 표준편차)')
ax.set_title('개별 모델 성능')
for i, v in enumerate(d['CV AUC']):
    ax.text(v, i, f'  {v:.6f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## 5. 다양성 측정 — 앙상블이 효과가 있을까

앙상블의 이득은 **모델들이 서로 다르게 틀릴 때** 생긴다.
OOF 예측끼리의 **상관계수**로 이를 확인할 수 있다.

| 상관계수 | 해석 |
|---|---|
| 0.99 이상 | 거의 같은 모델. 앙상블 이득이 거의 없다 |
| 0.95 ~ 0.99 | 적당한 다양성. **일반적인 부스팅 앙상블 구간** |
| 0.95 미만 | 다양성이 크다. 이득이 클 수 있지만, 한쪽이 약한 모델일 수도 있다 |

In [ ]:
oof_df = pd.DataFrame({k: v['oof'] for k, v in models.items()})
corr = oof_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(corr, annot=True, fmt='.5f', cmap='RdBu_r',
            vmin=0.9, vmax=1.0, square=True, ax=axes[0],
            cbar_kws={'shrink': 0.8})
axes[0].set_title('OOF 예측 상관관계 — 낮을수록 다양성이 크다')

# 순위(rank) 기준 상관도 함께 본다.
# AUC는 '값' 자체가 아니라 '순서'로 계산되므로 순위 상관이 더 직접적인 지표다.
rank_corr = oof_df.corr(method='spearman')
sns.heatmap(rank_corr, annot=True, fmt='.5f', cmap='RdBu_r',
            vmin=0.9, vmax=1.0, square=True, ax=axes[1],
            cbar_kws={'shrink': 0.8})
axes[1].set_title('순위(Spearman) 상관 — AUC와 직결되는 지표')

plt.tight_layout(); plt.show()

off_diag = corr.values[np.triu_indices_from(corr.values, k=1)]
print(f'모델 간 평균 상관: {off_diag.mean():.5f}')
print(f'가장 다른 조합: {corr.stack().drop_duplicates().nsmallest(1).index.tolist()}')

## 6. 블렌딩 — 세 가지 방법 비교

### 방법 1. 단순 평균 (Simple Average)
```
최종 = (LGB + XGB + CB) / 3
```
가장 단순하지만 놀랄 만큼 강하다. **과적합 위험이 없다**는 게 최대 장점이다.

### 방법 2. 순위 평균 (Rank Average)
각 모델의 예측을 **순위로 바꾼 뒤** 평균낸다.

왜 필요한가: 모델마다 확률값의 분포가 다르다. 한 모델은 0.9 근처에 몰려 있고
다른 모델은 0.6 근처에 퍼져 있으면, 단순 평균 시 **한쪽 모델이 결과를 지배**한다.
순위로 바꾸면 분포 차이가 사라져 **공평하게 섞인다.**
평가 지표가 AUC(순위 기반)일 때 특히 잘 맞는다.

### 방법 3. 가중 평균 (Weighted Average)
OOF 예측으로 **최적 가중치를 탐색**한다.
성능이 좋은 모델에 더 큰 가중치를 준다.

> ⚠️ **주의**: 가중치를 OOF에서 찾으면 OOF에 살짝 과적합된다.
> 그래서 가중 평균의 CV 점수는 **약간 부풀려진 값**으로 봐야 한다.
> 차이가 미미하면 **더 단순한 방법(단순/순위 평균)을 고르는 게 안전하다.**

In [ ]:
oofs = {k: v['oof'] for k, v in models.items()}
tests = {k: v['test_pred'] for k, v in models.items()}
names = list(models.keys())

blend = {}

# ── 방법 1: 단순 평균 ──
blend['단순 평균'] = dict(
    oof=np.mean([oofs[k] for k in names], axis=0),
    test=np.mean([tests[k] for k in names], axis=0),
)

# ── 방법 2: 순위 평균 ──
# rankdata: 값을 순위로 변환. 0~1로 정규화해 확률처럼 만든다.
def rank_norm(a):
    return rankdata(a) / len(a)

blend['순위 평균'] = dict(
    oof=np.mean([rank_norm(oofs[k]) for k in names], axis=0),
    test=np.mean([rank_norm(tests[k]) for k in names], axis=0),
)

for k, v in blend.items():
    print(f'{k:12s} OOF AUC = {roc_auc_score(y, v["oof"]):.6f}')

In [ ]:
# ── 방법 3: 가중치 탐색 ──
# 0.05 단위로 가능한 모든 조합(합=1)을 전부 시도한다.
# 모델이 3개뿐이라 완전탐색이 가능하다. 모델이 많으면 scipy.optimize를 쓴다.
from itertools import product

step = 0.05
grid = np.arange(0, 1 + step, step)
best_w, best_auc = None, 0

for w in product(grid, repeat=len(names)):
    if abs(sum(w) - 1.0) > 1e-9:
        continue
    p = sum(wi * oofs[k] for wi, k in zip(w, names))
    a = roc_auc_score(y, p)
    if a > best_auc:
        best_auc, best_w = a, w

print('최적 가중치:')
for k, w in zip(names, best_w):
    print(f'  {k:12s} {w:.2f}')
print(f'가중 평균 OOF AUC = {best_auc:.6f}')

blend['가중 평균'] = dict(
    oof=sum(wi * oofs[k] for wi, k in zip(best_w, names)),
    test=sum(wi * tests[k] for wi, k in zip(best_w, names)),
)

In [ ]:
# 개별 모델과 블렌딩 결과를 한 표로 정리
rows = [{'방법': k, '종류': '단일 모델', 'CV AUC': v['cv_auc']} for k, v in models.items()]
rows += [{'방법': k, '종류': '앙상블', 'CV AUC': roc_auc_score(y, v['oof'])}
         for k, v in blend.items()]

final = pd.DataFrame(rows).sort_values('CV AUC', ascending=False).reset_index(drop=True)
best_single = max(v['cv_auc'] for v in models.values())
final['최고 단일모델 대비'] = (final['CV AUC'] - best_single).round(6)
print(final.round(6).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
d = final.sort_values('CV AUC')
ax.barh(d['방법'], d['CV AUC'],
        color=['#fb8500' if t == '앙상블' else '#457b9d' for t in d['종류']])
ax.axvline(best_single, color='red', ls='--', lw=1.2, label='최고 단일 모델')
ax.set_xlim(d['CV AUC'].min() - 0.001, d['CV AUC'].max() + 0.0005)
ax.set_xlabel('CV AUC')
ax.set_title('단일 모델 vs 앙상블 (주황 = 앙상블)')
ax.legend(fontsize=8)
for i, v in enumerate(d['CV AUC']):
    ax.text(v, i, f'  {v:.6f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## 7. 최종 제출 파일 생성

In [ ]:
# 가장 점수가 높은 방법을 고른다.
# 단, 가중 평균이 OOF 과적합으로 유리해 보일 수 있으므로,
# 차이가 0.0001 미만이면 더 단순하고 안전한 방법을 택한다.
ens_only = final[final['종류'] == '앙상블'].sort_values('CV AUC', ascending=False)
top = ens_only.iloc[0]

if top['방법'] == '가중 평균' and len(ens_only) > 1:
    runner = ens_only.iloc[1]
    if top['CV AUC'] - runner['CV AUC'] < 0.0001:
        print(f"가중 평균이 1위지만 2위({runner['방법']})와의 차이가 "
              f"{top['CV AUC']-runner['CV AUC']:.6f}로 미미하다.")
        print(f"-> OOF 과적합 위험을 피해 더 단순한 '{runner['방법']}'을 채택한다.")
        top = runner

CHOSEN = top['방법']
final_pred = blend[CHOSEN]['test']
print(f"\n채택: {CHOSEN}  (OOF AUC {top['CV AUC']:.6f})")

In [ ]:
os.makedirs('../submissions', exist_ok=True)

submission = sub.copy()
submission[TARGET] = final_pred
submission.to_csv('../submissions/submission_v3_ensemble.csv', index=False)

# 제출 전 최종 점검 — 여기서 걸러야 할 사고가 많다
print('=== 제출 파일 점검 ===')
print(f'행 개수      : {len(submission):,}  (sample_submission {len(sub):,})')
print(f'열 이름      : {list(submission.columns)}')
print(f'id 일치      : {(submission["id"].values == sub["id"].values).all()}')
print(f'결측치       : {submission[TARGET].isnull().sum()}개')
print(f'값 범위      : {submission[TARGET].min():.6f} ~ {submission[TARGET].max():.6f}')
print(f'평균         : {submission[TARGET].mean():.6f}  (train 양성비율 {y.mean():.6f})')
print()
print(submission.head())

In [ ]:
# 예측 분포를 마지막으로 확인한다.
# OOF와 테스트 예측의 분포가 비슷해야 정상이다.
# 크게 다르면 train/test 분포 차이나 전처리 버그를 의심해야 한다.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.kdeplot(blend[CHOSEN]['oof'], ax=axes[0], fill=True,
            color='#457b9d', label='OOF (train)')
sns.kdeplot(final_pred, ax=axes[0], fill=True,
            color='#fb8500', label='최종 예측 (test)', alpha=0.5)
axes[0].set_title('OOF vs 테스트 예측 분포')
axes[0].set_xlabel('예측값'); axes[0].legend(fontsize=8)

for k in names:
    sns.kdeplot(tests[k], ax=axes[1], label=k, lw=1.5)
sns.kdeplot(final_pred, ax=axes[1], label=f'앙상블({CHOSEN})',
            lw=2.5, color='black', ls='--')
axes[1].set_title('모델별 테스트 예측 분포')
axes[1].set_xlabel('예측 확률'); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# 모든 예측 결과를 저장해둔다 (나중에 다른 조합을 실험할 때 재학습 불필요)
os.makedirs('../preds', exist_ok=True)
for k, v in models.items():
    np.save(f'../preds/oof_{k.lower()}.npy', v['oof'])
    np.save(f'../preds/test_{k.lower()}.npy', v['test_pred'])

with open('../preds/ensemble_summary.json', 'w', encoding='utf-8') as f:
    json.dump({
        'individual': {k: float(v['cv_auc']) for k, v in models.items()},
        'blend': {k: float(roc_auc_score(y, v['oof'])) for k, v in blend.items()},
        'best_weights': {k: float(w) for k, w in zip(names, best_w)},
        'chosen': CHOSEN,
    }, f, ensure_ascii=False, indent=2)

print('저장 완료: preds/*.npy, preds/ensemble_summary.json')
print('제출 파일: submissions/submission_v3_ensemble.csv')

---
## 📌 05단계 요약 & 대회 마무리

### 제출 방법
캐글에 제출하려면 터미널에서:

```powershell
kaggle competitions submit -c playground-series-s6e8 `
  -f s6e8-smartphone/submissions/submission_v3_ensemble.csv `
  -m "LGB+XGB+CatBoost ensemble"
```

제출 후 리더보드(LB) 점수를 받으면 **CV와 LB가 같은 방향으로 움직이는지**
확인하자. 방향이 어긋나면 검증 틀에 문제가 있다는 신호다.

### 배운 것 — 앙상블의 3원칙
1. **다양성이 이득의 원천이다.** 상관이 0.99를 넘으면 앙상블 효과가 거의 없다.
   모델 종류를 바꾸는 것이 시드만 바꾸는 것보다 훨씬 효과적이다.
2. **단순 평균은 강력하다.** 가중치를 최적화해도 이득이 미미할 때가 많고,
   OOF 과적합 위험까지 생긴다. 차이가 작으면 단순한 쪽을 택하라.
3. **AUC 평가에서는 순위 평균을 우선 검토하라.** 모델 간 확률 분포 차이를
   자동으로 해소해준다.

### 더 해볼 만한 것
- [ ] 시드를 바꿔 여러 번 학습 후 평균 (seed averaging)
- [ ] 스태킹 — OOF 예측을 입력으로 하는 2단계 메타 모델
- [ ] 신경망(MLP) 추가 — 트리 모델과 성격이 크게 달라 다양성 확보에 유리
- [ ] 타깃 인코딩 등 범주형 고급 처리